# US Michelin Starred Restaurants — Present Day Scrape
### Part 1 of the Michelin Inflation Project

**Note on the 403 error:** The Algolia API key from the original notebook (`3222e669cf...`) has been rotated or is now locked to requests coming from `guide.michelin.com` as a referer. This is common — Michelin periodically cycles these.  

**Two ways to fix this:**
1. **(Recommended) Grab a fresh key yourself:** Open `guide.michelin.com/us/en/restaurants` in Chrome, open DevTools → Network tab, filter by XHR/Fetch, search the page, and look for a request to `algolia.net`. The new `x-algolia-api-key` will be in the request headers. Paste it into `ALGOLIA_API_KEY` below and Section 2 will work.
2. **Use the HTML scraper in Section 3** — no key needed, scrapes the rendered pages directly with BeautifulSoup. Slower but totally self-contained.

Both approaches produce the same output CSV.

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import json
import re
from datetime import datetime
from bs4 import BeautifulSoup

---
## APPROACH A — Algolia API (fast, ~2 min for full dataset)
Only works if you have a valid key. See the note at the top for how to get one.

---

In [ ]:
# paste fresh key here — grab it from the network tab on guide.michelin.com
# the application ID (8NVHRD7ONV) hasn't changed, just the api key rotates
ALGOLIA_API_KEY = "3222e669cf890dc73fa5f38241117ba5&"
ALGOLIA_APP_ID  = "8NVHRD7ONV"

ALGOLIA_URL = (
    f"https://{ALGOLIA_APP_ID.lower()}-dsn.algolia.net/1/indexes/*/queries"
    f"?x-algolia-agent=Algolia%20for%20JavaScript%20(5.47.0)%3B%20Lite%20(5.47.0)%3B%20Browser"
    f"&x-algolia-api-key={ALGOLIA_API_KEY}"
    f"&x-algolia-application-id={ALGOLIA_APP_ID}"
)

# the referer header is the key thing — Michelin's Algolia key is locked to requests
# that look like they're coming from the actual site
ALGOLIA_HEADERS = {
    "Content-Type": "application/json",
    "Referer": "https://guide.michelin.com/",
    "Origin": "https://guide.michelin.com",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
}

PAGE_SIZE = 48

In [ ]:
def build_algolia_query(page_num, hits_per_page=PAGE_SIZE):
    return {
        "requests": [
            {
                "indexName": "prod-restaurants-en",
                "aroundLatLngViaIP": True,
                "aroundRadius": "all",
                "attributesToHighlight": [],
                "attributesToRetrieve": [
                    "_geoloc", "region", "area_name", "chef", "city", "country",
                    "cuisines", "currency", "good_menu", "identifier",
                    "michelin_award", "name", "slug", "new_table", "offers",
                    "online_booking", "other_urls", "region_code", "site_slug",
                    "site_name", "take_away", "delivery", "price_category",
                    "currency_symbol", "url", "green_star", "objectType"
                ],
                "filters": "status:Published",
                "hitsPerPage": hits_per_page,
                "optionalFilters": ["sites:us"],
                "page": page_num,
                "query": ""
            }
        ]
    }

In [ ]:
def parse_restaurant_algolia(r):
    """Same parser as before, handles None gracefully."""
    geo       = r.get("_geoloc") or {}
    city      = r.get("city") or {}
    country   = r.get("country") or {}
    pricing   = r.get("price_category") or {}
    region    = r.get("region") or {}
    cuisines  = r.get("cuisines") or []
    greenStar = r.get("green_star")
    if isinstance(greenStar, dict):
        greenStar = greenStar.get("label")
    
    return {
        "Name":            r.get("name"),
        "Award":           r.get("michelin_award"),
        "Food Style 1":    cuisines[0]["label"] if len(cuisines) > 0 else "NA",
        "Food Style 2":    cuisines[1]["label"] if len(cuisines) > 1 else "NA",
        "Pricing":         pricing.get("slug"),
        "Online Booking?": r.get("online_booking"),
        "Take Away?":      r.get("take_away"),
        "Delivery?":       r.get("delivery"),
        "Chef":            r.get("chef"),
        "City":            city.get("name"),
        "Area":            r.get("area_name"),
        "Region":          region.get("name"),
        "Country":         country.get("name"),
        "Country Code":    country.get("code"),
        "Currency":        r.get("currency"),
        "Good Menu?":      r.get("good_menu"),
        "Green Star?":     greenStar,
        "Latitude":        geo.get("lat"),
        "Longitude":       geo.get("lng"),
        "Michelin URL":    r.get("url"),
        "Identifier":      r.get("identifier")
    }

In [ ]:
# quick test — run this before the full scrape to see if the key is valid
# if you still get 403 here, skip to Approach B below
probe = requests.post(ALGOLIA_URL, headers=ALGOLIA_HEADERS, json=build_algolia_query(0))
print(f"Status: {probe.status_code}")

if probe.status_code == 200:
    probeData   = probe.json()
    firstResult = probeData["results"][0]
    totalHits   = firstResult["nbHits"]
    totalPages  = firstResult["nbPages"]
    print(f"Total restaurants: {totalHits}")
    print(f"Total pages:       {totalPages}")
    print("Key works — run the Algolia scraper below.")
else:
    print("Still getting blocked. Jump to Approach B (HTML scraper).")
    print(f"Response: {probe.text[:200]}")

In [8]:
# only run this cell if the probe above came back 200

algoliaRestaurants = []
failedPages        = []

print(f"Scraping {totalPages} pages via Algolia...")

for page in range(totalPages):
    try:
        r    = requests.post(ALGOLIA_URL, headers=ALGOLIA_HEADERS, json=build_algolia_query(page))
        hits = r.json()["results"][0]["hits"]
        for restaurant in hits:
            algoliaRestaurants.append(parse_restaurant_algolia(restaurant))
        if page % 5 == 0:
            print(f"  Page {page}/{totalPages-1} — {len(algoliaRestaurants)} restaurants so far")
        time.sleep(0.5)
    except Exception as e:
        print(f"  Page {page} failed: {e}")
        failedPages.append(page)
        time.sleep(1)

print(f"Done. {len(algoliaRestaurants)} restaurants collected.")
if failedPages:
    print(f"Failed pages: {failedPages}")

Scraping 393 pages via Algolia...
  Page 0/392 — 48 restaurants so far
  Page 5/392 — 288 restaurants so far
  Page 10/392 — 528 restaurants so far
  Page 15/392 — 768 restaurants so far
  Page 20/392 — 1008 restaurants so far
  Page 25/392 — 1248 restaurants so far
  Page 30/392 — 1488 restaurants so far
  Page 35/392 — 1728 restaurants so far
  Page 40/392 — 1968 restaurants so far
  Page 45/392 — 2208 restaurants so far
  Page 50/392 — 2448 restaurants so far
  Page 55/392 — 2688 restaurants so far
  Page 60/392 — 2928 restaurants so far
  Page 65/392 — 3168 restaurants so far
  Page 70/392 — 3408 restaurants so far
  Page 75/392 — 3648 restaurants so far
  Page 80/392 — 3888 restaurants so far
  Page 85/392 — 4128 restaurants so far
  Page 90/392 — 4368 restaurants so far
  Page 95/392 — 4608 restaurants so far
  Page 100/392 — 4848 restaurants so far
  Page 105/392 — 5088 restaurants so far
  Page 110/392 — 5328 restaurants so far
  Page 115/392 — 5568 restaurants so far
  Page 12

---
## APPROACH B — HTML Scraper (no API key needed)
Scrapes `guide.michelin.com` directly. Slower (~15-20 min for the full US dataset) but doesn't depend on Algolia keys.  
Uses BeautifulSoup to parse the restaurant cards from each paginated results page.

---

In [10]:
# browser-like headers — guide.michelin.com returns near-empty HTML without these
HTML_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
}

# base URL — we'll page through ?page=1, ?page=2, etc.
BASE_URL = "https://guide.michelin.com/us/en/restaurants"

# we want starred + bib gourmand + selected — this gets everything
# can filter later if we only want stars

In [11]:
def get_total_pages(base_url):
    """Fetches page 1 and reads the pagination to find the total page count."""
    r = requests.get(base_url, headers=HTML_HEADERS)
    if r.status_code != 200:
        print(f"Couldn't load page 1 — status {r.status_code}")
        return None
    
    soup = BeautifulSoup(r.text, "html.parser")
    
    # Michelin's pagination is in a <ul class="pagination"> — last numbered item is total pages
    pagination = soup.select("ul.pagination li a")
    pageNums = []
    for link in pagination:
        try:
            pageNums.append(int(link.text.strip()))
        except ValueError:
            pass  # skip 'Next' / 'Prev' buttons
    
    if pageNums:
        return max(pageNums)
    else:
        # fallback: look for the total count in the results header
        countTag = soup.select_one(".search-results__header")
        if countTag:
            print(f"Results header: {countTag.text.strip()}")
        print("Couldn't auto-detect page count — check the HTML structure")
        return None

In [12]:
def parse_restaurant_html(card):
    """
    Parses a single restaurant card from the HTML.
    Michelin uses div.card__menu for each listing.
    Fields available in HTML are fewer than Algolia — no lat/long, no booking flags,
    but we get name, award, cuisine, city, and URL which is what we need most.
    """
    
    # name
    nameTag = card.select_one(".card__menu-content--title h3") or card.select_one(".card__menu-content h3")
    name    = nameTag.text.strip() if nameTag else None
    
    # award — stored in a data attribute or an image alt text
    awardTag = card.select_one("[data-v-sticky-header] .card__tag") or card.select_one(".card__menu-content--title .tag")
    award    = awardTag.text.strip() if awardTag else None
    
    # if award didn't parse from tag, try the michelin distinction icon alt text
    if not award:
        iconTag = card.select_one(".card__menu-footer--awards img")
        if iconTag:
            award = iconTag.get("alt", "").strip()
    
    # cuisine type
    cuisineTag = card.select_one(".card__menu-content--type")
    cuisineRaw = cuisineTag.text.strip() if cuisineTag else ""
    # usually comes as "Italian · Contemporary" — split on ·
    cuisineParts = [c.strip() for c in cuisineRaw.split("·")]
    cuisineOne   = cuisineParts[0] if len(cuisineParts) > 0 else "NA"
    cuisineTwo   = cuisineParts[1] if len(cuisineParts) > 1 else "NA"
    
    # location — city and region often in a subtitle or address field
    locationTag = card.select_one(".card__menu-content--city") or card.select_one(".card__menu-content--location")
    location    = locationTag.text.strip() if locationTag else None
    
    # price level — usually shown as $, $$, $$$, $$$$
    priceTag  = card.select_one(".card__menu-content--price")
    priceRaw  = priceTag.text.strip() if priceTag else None
    
    # URL slug for later matching with Google/TripAdvisor
    linkTag   = card.select_one("a.card__menu")
    if not linkTag:
        linkTag = card.find("a", href=True)
    michelinURL = "https://guide.michelin.com" + linkTag["href"] if linkTag else None
    
    return {
        "Name":         name,
        "Award":        award,
        "Food Style 1": cuisineOne,
        "Food Style 2": cuisineTwo,
        "Location":     location,
        "Pricing":      priceRaw,
        "Country":      "USA",
        "Michelin URL": michelinURL
    }

In [13]:
# figure out how many pages exist before committing to the loop
totalPages = get_total_pages(BASE_URL)
print(f"Total pages detected: {totalPages}")

# if auto-detect failed, set it manually — usually ~65-70 pages for US
# totalPages = 68

Total pages detected: 393


In [14]:
htmlRestaurants = []
failedPages     = []

print(f"Starting HTML scrape — {totalPages} pages")
print(f"Started: {datetime.now().strftime('%H:%M:%S')}")
print("-" * 40)

for page in range(1, totalPages + 1):
    try:
        url = f"{BASE_URL}?page={page}"
        r   = requests.get(url, headers=HTML_HEADERS)
        
        if r.status_code != 200:
            print(f"  Page {page}: {r.status_code}")
            failedPages.append(page)
            time.sleep(2)
            continue
        
        soup  = BeautifulSoup(r.text, "html.parser")
        # restaurant cards — Michelin uses .card__menu as the container
        cards = soup.select("div.card--restaurant") or soup.select(".card__menu")
        
        if not cards:
            print(f"  Page {page}: no cards found — HTML structure may have changed")
            # dump the first 500 chars to help debug
            print(f"  Preview: {r.text[:300]}")
            failedPages.append(page)
        
        for card in cards:
            parsed = parse_restaurant_html(card)
            if parsed["Name"]:  # skip empties
                htmlRestaurants.append(parsed)
        
        if page % 10 == 0:
            print(f"  Page {page}/{totalPages} — {len(htmlRestaurants)} collected")
        
        # being polite — HTML scraping needs a longer sleep than API calls
        time.sleep(1.5)
        
    except Exception as e:
        print(f"  Page {page} blew up: {e}")
        failedPages.append(page)
        time.sleep(3)

print("-" * 40)
print(f"Done at {datetime.now().strftime('%H:%M:%S')}")
print(f"Total restaurants: {len(htmlRestaurants)}")
if failedPages:
    print(f"Failed pages: {failedPages}")

Starting HTML scrape — 393 pages
Started: 15:35:13
----------------------------------------
  Page 10/393 — 520 collected


KeyboardInterrupt: 

---
## 4. Build DataFrame + Sanity Checks
Works whether you used Approach A or B — just swap the variable name.

---

In [20]:
# swap this to algoliaRestaurants if Approach A worked
rawData = algoliaRestaurants

michelinUS = pd.DataFrame(rawData)
print(f"Shape: {michelinUS.shape}")
michelinUS.head(10)

Shape: (18837, 21)


,Name,Award,Food Style 1,Food Style 2,Pricing,Online Booking?,Take Away?,Delivery?,Chef,City,...,Region,Country,Country Code,Currency,Good Menu?,Green Star?,Latitude,Longitude,Michelin URL,Identifier
0,La'Shukran,selected,Middle Eastern,NA,premium,0,0,0,None,Washington,...,District of Columbia,USA,US,USD,0,None,38.907295,-76.999608,/us/en/district-of-columbia/washington-dc/rest...,1216336
1,Café Riggs,selected,Contemporary,French,premium,1,0,0,None,Washington,...,District of Columbia,USA,US,USD,0,None,38.897556,-77.024263,/us/en/district-of-columbia/washington-dc/rest...,1190829
2,Xiquet,ONE_STAR,Spanish,NA,luxury,1,0,0,None,Washington,...,District of Columbia,USA,US,USD,0,None,38.921587,-77.072383,/us/en/district-of-columbia/washington-dc/rest...,1184921
3,Rooster & Owl,ONE_STAR,Contemporary,Fusion,premium,1,0,0,None,Washington,...,District of Columbia,USA,US,USD,0,None,38.921467,-77.032160,/us/en/district-of-columbia/washington-dc/rest...,1164753
4,Family Ethiopian,selected,Ethiopian,Regional Cuisine,mid-range,0,0,0,None,Washington,...,District of Columbia,USA,US,USD,0,None,38.909282,-77.024210,/us/en/district-of-columbia/washington-dc/rest...,1195106
5,Masseria,ONE_STAR,Italian,Regional Cuisine,luxury,1,0,0,None,Washington,...,District of Columbia,USA,US,USD,0,None,38.909504,-76.999080,/us/en/district-of-columbia/washington-dc/rest...,499964
6,Ris,selected,American,NA,mid-range,1,0,0,None,Washington,...,District of Columbia,USA,US,USD,0,None,38.903840,-77.049930,/us/en/district-of-columbia/washington-dc/rest...,499989
7,Thip Khao,selected,Lao,NA,mid-range,1,0,0,None,Washington,...,District of Columbia,USA,US,USD,0,None,38.932957,-77.032845,/us/en/district-of-columbia/washington-dc/rest...,500001
8,Toki Underground,BIB_GOURMAND,Japanese,Ramen,affordable,1,0,0,None,Washington,...,District of Columbia,USA,US,USD,0,None,38.900337,-76.988910,/us/en/district-of-columbia/washington-dc/rest...,502400
9,Ellē,BIB_GOURMAND,Contemporary,Bakery,mid-range,1,0,0,None,Washington,...,District of Columbia,USA,US,USD,0,None,38.931870,-77.038420,/us/en/district-of-columbia/washington-dc/rest...,561986


In [24]:
print("Award breakdown:")
print(michelinUS["Award"].value_counts())

Award breakdown:
Award
selected        11481
BIB_GOURMAND     3539
ONE_STAR         3127
TWO_STARS         534
THREE_STARS       156
Name: count, dtype: int64


In [26]:
print("Top 20 cuisines:")
print(michelinUS["Food Style 1"].value_counts().head(20))

Top 20 cuisines:
Food Style 1
Modern Cuisine           3569
Creative                 1046
Contemporary             1010
Traditional Cuisine       910
Japanese                  800
Italian                   638
Seafood                   618
French                    517
Mediterranean Cuisine     450
Farm to table             403
Regional Cuisine          377
Classic Cuisine           359
Country cooking           329
Modern British            315
Street Food               276
Modern French             263
Mexican                   256
International             244
Thai                      236
Cantonese                 228
Name: count, dtype: int64


In [22]:
print("Null counts:")
print(michelinUS.isnull().sum())

Null counts:
Name                   0
Award                  0
Food Style 1           0
Food Style 2           0
Pricing                0
Online Booking?        0
Take Away?             0
Delivery?              0
Chef               18131
City                   0
Area               14398
Region                 0
Country                0
Country Code           0
Currency               0
Good Menu?             0
Green Star?        18837
Latitude               0
Longitude              0
Michelin URL           0
Identifier             0
dtype: int64


## 5. Save

In [28]:
michelinUS["Scrape Date"] = datetime.now().strftime("%Y-%m-%d")
michelinUS["Market"]      = "US"

michelinUS.to_csv("michelin_us_full_current.csv", index=False)
print(f"Saved: michelin_us_full_current.csv ({len(michelinUS)} rows)")

# starred only — filter depends on which approach was used
# Algolia uses 'ONE_STAR' / 'TWO_STARS' / 'THREE_STARS'
# HTML scraper may use '1 Star' / '2 Stars' / '3 Stars' depending on what Michelin renders
starKeywords = ["star", "Star", "ONE_STAR", "TWO_STARS", "THREE_STARS"]
starredOnly  = michelinUS[michelinUS["Award"].fillna("").str.contains("|".join(starKeywords))].copy()
starredOnly.to_csv("michelin_us_starred_current.csv", index=False)
print(f"Saved: michelin_us_starred_current.csv ({len(starredOnly)} rows)")

Saved: michelin_us_full_current.csv (18837 rows)
Saved: michelin_us_starred_current.csv (3817 rows)


In [30]:
print("=" * 50)
print("SCRAPE SUMMARY — US MICHELIN GUIDE (CURRENT)")
print("=" * 50)
print(f"  Scrape date:       {michelinUS['Scrape Date'].iloc[0]}")
print(f"  Total restaurants: {len(michelinUS)}")
print(f"  Starred:           {len(starredOnly)}")
print(f"  Unique cuisines:   {michelinUS['Food Style 1'].nunique()}")
print("=" * 50)
print()
print("Next: same notebook for UK, France, Canada, EU.")
print("After that: historical comparison going back to 2015.")

SCRAPE SUMMARY — US MICHELIN GUIDE (CURRENT)
  Scrape date:       2026-03-17
  Total restaurants: 18837
  Starred:           3817
  Unique cuisines:   262

Next: same notebook for UK, France, Canada, EU.
After that: historical comparison going back to 2015.
